# License Plate Bbox Detector -- YOLOv8n Training

Trains YOLOv8n on the Open Images V7 "Vehicle registration plate" class (3000 train, 500 val).
No account needed. Uses Colab T4 GPU.

**Before running:** Runtime -> Change runtime type -> T4 GPU

## References

- Redmon et al., *You Only Look Once: Unified, Real-Time Object Detection*, CVPR 2016
- Redmon & Farhadi, *YOLOv3: An Incremental Improvement*, arXiv:1804.02767
- Bochkovskiy et al., *YOLOv4: Optimal Speed and Accuracy of Object Detection*, arXiv:2004.10934
- Jocher et al., *Ultralytics YOLOv8*, 2023, https://github.com/ultralytics/ultralytics
- Kuznetsova et al., *The Open Images Dataset V4*, IJCV 2020 -- dataset source

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics fiftyone

## 2. Download plates from Open Images V7

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

train_dataset = foz.load_zoo_dataset(
    'open-images-v7',
    split='train',
    label_types=['detections'],
    classes=['Vehicle registration plate'],
    max_samples=3000,
    dataset_name='plates_train',
)

val_dataset = foz.load_zoo_dataset(
    'open-images-v7',
    split='validation',
    label_types=['detections'],
    classes=['Vehicle registration plate'],
    max_samples=500,
    dataset_name='plates_val',
)

print(f'Train samples: {len(train_dataset)}')
print(f'Val samples:   {len(val_dataset)}')

## 3. Convert to YOLO format

In [ ]:
import os

DATASET_DIR = '/content/plate_dataset'

# Open Images uses 'ground_truth' as the field name, not 'detections'
sample = train_dataset.first()
label_field = None
for field_name, field in sample.iter_fields():
    if hasattr(field, 'detections'):
        label_field = field_name
        break

if label_field is None:
    print('Available fields:', list(sample.field_names))
    raise ValueError('Could not find a detections field.')

print(f'Using label field: {label_field!r}')

train_dataset.export(
    export_dir=os.path.join(DATASET_DIR, 'train'),
    dataset_type=fo.types.YOLOv5Dataset,
    label_field=label_field,
    classes=['Vehicle registration plate'],
)

val_dataset.export(
    export_dir=os.path.join(DATASET_DIR, 'val'),
    dataset_type=fo.types.YOLOv5Dataset,
    label_field=label_field,
    classes=['Vehicle registration plate'],
)

print('Exported to:', DATASET_DIR)

In [ ]:
import yaml

yaml_content = f'path: {DATASET_DIR}\ntrain: train/images\nval: val/images\n\nnc: 1\nnames: [plate]\n'
yaml_path = os.path.join(DATASET_DIR, 'dataset.yaml')

with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(yaml_content)

## 4. Train YOLOv8n

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project='/content/runs',
    name='plate_detector',
    patience=10,
    exist_ok=True,
)

print('Best weights:', results.save_dir)

## 5. Evaluate

In [ ]:
best_weights = '/content/runs/plate_detector/weights/best.pt'
trained = YOLO(best_weights)
metrics = trained.val(data=yaml_path)

print(f'mAP50:    {metrics.box.map50:.3f}')
print(f'mAP50-95: {metrics.box.map:.3f}')

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
                   all        500        680      0.925      0.835       0.89      0.619
mAP50:    0.890
mAP50-95: 0.619


## 6. Export to ONNX

In [ ]:
trained.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    opset=12,
)

onnx_path = best_weights.replace('.pt', '.onnx')
print('ONNX exported to:', onnx_path)

## 7. Save to Google Drive

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
shutil.copy(onnx_path, '/content/drive/MyDrive/plate-detector.onnx')
print('Done. Copy to: frontend/public/models/plate-detector.onnx')